# CT cylinder-crop examples

This recipe writes separate automatic and manual cylindrical crops from an HDF5 CT volume. Continue with [notebook 03](03_ct_porosity_analysis_and_visualisation.ipynb) for read-only porespace analysis.

In [ ]:
from pathlib import Path
import sys

import h5py
import numpy as np

working_directory = Path.cwd().resolve()
project_root = next(
    (
        candidate
        for candidate in (working_directory, *working_directory.parents)
        if (candidate / "pyproject.toml").is_file()
    ),
    None,
)
if project_root is None:
    raise RuntimeError("Run this notebook from within the basalt-processing project.")

source_root = project_root / "src"
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

from basalt_processing.cylinder_crop import (
    build_cylinder_mask,
    detect_circle_on_slice,
    fit_cylinder_parameters,
    write_cylinder_crop,
)
from basalt_processing.paths import load_config, resolve_path

CONFIG_PATH = project_root / "config" / "basalt.example.toml"
config = load_config(CONFIG_PATH)
paths_config = config.get("paths", {})
data_root = resolve_path(paths_config["data_root"], config.get("_config_dir"))

SOURCE_H5_DATA = resolve_path(
    "CT/pre-test/processed/"
    "20200275-CYL11159231-1500-100kV-200uA-025mmAg_with_foam.hdf5",
    data_root,
)
SOURCE_H5_BINARY = resolve_path(
    "CT/pre-test/processed/"
    "20200275-CYL11159231-1500-100kV-200uA-025mmAg_with_foam(2).hdf5",
    data_root,
)
OUTPUT_H5 = resolve_path("output/data_cylinder_crop.hdf5", data_root)
MANUAL_OUTPUT_H5 = OUTPUT_H5.with_stem(f"{OUTPUT_H5.stem}_manual")
DATA_DATASET = "data"
THRESHOLD_DATASET = "threshold_mask"
MASK_DATASET = "cylinder_mask"
MASKED_DATASET = "data_masked"

## Spatial provenance of `foam(2)`

`20200275-CYL11159231-1500-100kV-200uA-025mmAg_with_foam(2).hdf5` uses the grid of a subvolume extracted from the original `20200275-CYL11159231-1500-100kV-200uA-025mmAg_with_foam.hdf5` volume. In Z-Y-X array order, the relationship is:

```python
foam2_region = original_data[350:1450, 500:1500, 500:1500]  # Z, Y, X
```

This produces shape `(1100, 1000, 1000)`. With voxel spacing `0.0405326 mm`, local voxel `foam2_region[0, 0, 0]` corresponds to original voxel `[350, 500, 500]` and physical XYZ offset `(20.2663, 20.2663, 14.1864) mm`. The original HDF5 has no explicit origin attribute, so retain this mapping whenever the two volumes are combined. This documentation cell does not execute the crop or modify either HDF5 file.

## Load and validate the source datasets

The source file contains both the CT `data` and its `threshold_mask`. They must be matching three-dimensional Z-Y-X volumes. Cylinder centres use X-Y coordinates. Each crop output contains `data`, `cylinder_mask`, and `data_masked`.

In [ ]:
with h5py.File(SOURCE_H5_BINARY, "r") as source_file:
    data_volume = source_file[DATA_DATASET]
    threshold_volume = source_file[THRESHOLD_DATASET]
    if data_volume.ndim != 3 or threshold_volume.ndim != 3:
        raise ValueError("data and threshold datasets must be 3-D Z-Y-X volumes")
    data_shape = data_volume.shape
    if threshold_volume.shape != data_shape:
        raise ValueError("data and threshold datasets must have matching Z-Y-X shapes")

    sample_z = np.linspace(0, data_shape[0] - 1, num=min(12, data_shape[0]), dtype=int)
    detections = []
    for z_index in sample_z:
        circle = detect_circle_on_slice(threshold_volume[z_index])
        if circle is not None:
            x, y, radius = circle
            detections.append((z_index, x, y, radius))

## Automatic Hough-circle workflow

Use OpenCV Hough-circle detection sampled above to fit a straight centreline, then write a crop derived from the data volume. Inspect detections before relying on an automatic crop.

In [ ]:
parameters = fit_cylinder_parameters(np.asarray(detections), z_size=data_shape[0])
automatic_mask = build_cylinder_mask(
    data_shape, radius=parameters.radius, centers_xy=parameters.centerline_xy
)
write_cylinder_crop(
    SOURCE_H5_BINARY, OUTPUT_H5, cylinder_mask=automatic_mask, data_dataset=DATA_DATASET,
    mask_dataset=MASK_DATASET, masked_dataset=MASKED_DATASET,
)

## Automatic crop output

The automatic crop was written to this separate output path:

In [ ]:
OUTPUT_H5

import matplotlib.pyplot as plt

with (
    h5py.File(SOURCE_H5_DATA, "r") as data_file,
    h5py.File(SOURCE_H5_BINARY, "r") as binary_file,
    h5py.File(OUTPUT_H5, "r") as output_file,
):
    mask = output_file[MASK_DATASET]
    z = mask.shape[0] // 2
    data_slice = data_file[DATA_DATASET][z + 350, 500:1500, 500:1500]
    threshold_slice = binary_file[THRESHOLD_DATASET][z]
    mask_slice = mask[z]
    if not (data_slice.shape == threshold_slice.shape == mask_slice.shape):
        raise ValueError("aligned data, threshold-mask, and cylinder-mask slices must match")
    masked_slice = np.where(mask_slice, data_slice, np.nan)

vmin, vmax = np.percentile(data_slice, [1, 99])

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(data_slice, cmap="gray", vmin=vmin, vmax=vmax)
axes[0].set_title(f"Original ({DATA_DATASET}) z={z}")
axes[0].axis("off")

axes[1].imshow(threshold_slice, cmap="gray")
axes[1].set_title(f"Binary mask ({THRESHOLD_DATASET}) z={z}")
axes[1].axis("off")

axes[2].imshow(mask_slice, cmap="gray")
axes[2].set_title(f"Cylinder mask ({MASK_DATASET}) z={z}")
axes[2].axis("off")

axes[3].imshow(masked_slice, cmap="gray", vmin=vmin, vmax=vmax)
axes[3].set_title(f"Masked original ({MASKED_DATASET}) z={z}")
axes[3].axis("off")

plt.tight_layout()
plt.show()

## Manual centreline and radius workflow

When automatic detections are unreliable, set a reviewed X-Y centre and radius in voxel coordinates. This writes a distinct output so it does not overwrite the automatic result.

In [ ]:
MANUAL_CENTER_XY = (1024.0, 1024.0)  # reviewed X, Y voxel centre
MANUAL_RADIUS = 900.0  # reviewed radius in voxels

manual_mask = build_cylinder_mask(
    data_shape, radius=MANUAL_RADIUS, center_xy=MANUAL_CENTER_XY
)
write_cylinder_crop(
    SOURCE_H5_BINARY, MANUAL_OUTPUT_H5, cylinder_mask=manual_mask, data_dataset=DATA_DATASET,
    mask_dataset=MASK_DATASET, masked_dataset=MASKED_DATASET,
)